In [ ]:
import os,re
import pandas as pd
from pyspark.sql import SparkSession, functions as f

In [ ]:
# create spark session
# spark = SparkSession.builder.appName("snack-analysis").getOrCreate()

spark = SparkSession.builder \
    .appName("snack-analysis") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

spark.conf.set("spark.sql.constraintPropagation.enabled", "false")

In [ ]:
# creating pandas dataframes from excel file
mcd_menu_xl = pd.read_excel('raw_data/menu.xlsx', dtype=str)
sb_nutrition_drinks_xl = pd.read_excel('raw_data/starbucks-menu-nutrition-drinks.xlsx', dtype=str)
sb_nutrition_foods_xl = pd.read_excel('raw_data/starbucks-menu-nutrition-food.xlsx', dtype=str)
# sb_drinks_xl = pd.read_excel("raw_data/starbucks_drinkMenu_expanded.xlsx", dtype=str)

In [ ]:
# creating pyspark dataframes
mcd_df = spark.createDataFrame(mcd_menu_xl)
sb_nutrition_drinks_df = spark.createDataFrame(sb_nutrition_drinks_xl)
sb_nutrition_foods_df = spark.createDataFrame(sb_nutrition_foods_xl)
# sb_drink_menu_df = spark.createDataFrame(sb_drinks_xl)

In [ ]:
# counting the number of "-" in nutrition drinks
sb_nutrition_drinks_df.select([
    f.sum(f.when(f.col(f"`{c}`") == "-", 1).otherwise(0)).alias(c)
    for c in sb_nutrition_drinks_df.columns
]).show()

In [ ]:
# generic funtion for displaying the dataframe
def inspect_dataframe(df, name):
    print(f"\nDataset: {name}")
    print(f"Rows: {df.count()}")
    print(f"Columns: {len(df.columns)}")

    df.show(5)
    df.printSchema()

In [ ]:
# renaming the columns
def rename_columns(df):
    if "Unnamed: 0" in df.columns:
        df = df.withColumnRenamed("Unnamed: 0", "Item")
    new_cols = ["_".join(re.sub(r"[()\.]", "", col).split()) for col in df.columns]
    df = df.toDF(*new_cols)
    return df

mcd_df = rename_columns(mcd_df)
sb_nutrition_drinks_df = rename_columns(sb_nutrition_drinks_df)
sb_nutrition_foods_df = rename_columns(sb_nutrition_foods_df)
# sb_drink_menu_df = rename_columns(sb_drink_menu_df)


In [ ]:
# converting the nutrient columns into numbers

def convert_nutrients_to_numbers(df, cols):
    return df.select([
        f.col(c).try_cast("double").alias(c)
        if c in cols
        else f.col(c)
        for c in df.columns
    ])

mcd_df = convert_nutrients_to_numbers(mcd_df, mcd_df.columns[3:])
sb_nutrition_foods_df = convert_nutrients_to_numbers(sb_nutrition_foods_df, sb_nutrition_foods_df.columns[1:])
sb_nutrition_drinks_df = convert_nutrients_to_numbers(sb_nutrition_drinks_df, sb_nutrition_drinks_df.columns[1:])
# sb_drink_menu_df = convert_nutrients_to_numbers(sb_drink_menu_df, sb_drink_menu_df.columns[3:])


In [ ]:
# counting number of nulls in each df
def count_nulls(df):
    num_cols = [name for name, type_ in df.dtypes if type_ == "double"]

    df.select([
        f.sum(
            f.when(f.col(c).isNull() | f.isnan(c), 1)
            .otherwise(0)
        ).alias(c)
        for c in num_cols
    ]).show()


# count_nulls(mcd_df)
# count_nulls(sb_nutrition_foods_df)
# count_nulls(sb_nutrition_drinks_df)
# count_nulls(sb_drink_menu_df)

In [ ]:
# removing empty rows
def remove_empty_calorie_rows(df, cols_to_remove):
    df = df.dropna(how="all", subset=[*cols_to_remove])
    return df

sb_nutrition_drinks_df = remove_empty_calorie_rows(sb_nutrition_drinks_df, sb_nutrition_drinks_df.columns[1:])

In [ ]:
# removing duplicates
def remove_duplicates(df):
    before = df.count()
    df = df.dropDuplicates()
    after = df.count()
    print("Duplicates removed:", before - after)
    return df
        
sb_nutrition_drinks_df = remove_duplicates(sb_nutrition_drinks_df)

In [ ]:
# creating a standard schema for all the dataframes

sb_food_standard_df = sb_nutrition_foods_df.select(
    f.col("Item").alias("item"),
    f.lit("Food").alias("category"),
    f.col("Calories").alias("calories"),
    f.col("Fat_g").alias("fat_g"),
    f.col("Carb_g").alias("carbs_g"),
    f.col("Fiber_g").alias("fiber_g"),
    f.col("Protein_g").alias("protein_g"),
    f.lit(None).cast("double").alias("sodium_mg"),
    f.lit(None).cast("double").alias("sugar_g"),
    f.lit("Starbucks").alias("brand"),
    f.lit("food").alias("kind")
)


sb_drinks_standard_df = sb_nutrition_drinks_df.select(
    f.col("Item").alias("item"),
    f.lit("Drinks").alias("category"),
    f.col("Calories").alias("calories"),
    f.col("Fat_g").alias("fat_g"),
    f.col("Carb_g").alias("carbs_g"),
    f.col("Fiber_g").alias("fiber_g"),
    f.col("Protein").alias("protein_g"),
    f.col("Sodium").alias("sodium_mg"),
    f.lit(None).cast("double").alias("sugar_g"),
    f.lit("Starbucks").alias("brand"),
    f.lit("drink").alias("kind")
)

drink_categories = [
    "Beverages",
    "Coffee & Tea",
    "Smoothies & Shakes"
]

mcd_standard_df = mcd_df.select(
    f.col("Item").alias("item"),
    f.col("Category").alias("category"),
    f.col("Calories").alias("calories"),
    f.col("Total_Fat").alias("fat_g"),
    f.col("Carbohydrates").alias("carbs_g"),
    f.col("Dietary_Fiber").alias("fiber_g"),
    f.col("Protein").alias("protein_g"),
    f.col("Sodium").alias("sodium_mg"),
    f.col("Sugars").alias("sugar_g"),
    f.lit("McDonald's").alias("brand"),
    f.when(
        f.lower(f.trim(f.col("Category"))).isin(
            [c.lower() for c in drink_categories]
        ),
        "drink"
    ).otherwise("food").alias("kind")
)

In [ ]:
sb_food_standard_df.printSchema()
sb_drinks_standard_df.printSchema()
mcd_standard_df.printSchema()

In [38]:
# combining the 3 datasets
all_nutrition_df = (
    mcd_standard_df
    .unionByName(sb_drinks_standard_df)
    .unionByName(sb_food_standard_df)
)

print("Total rows:", all_nutrition_df.count())
all_nutrition_df.show(10, truncate=False)
all_nutrition_df.printSchema()

Total rows: 448
+-------------------------------------------------------------+---------+--------+-----+-------+-------+---------+---------+-------+----------+----+
|item                                                         |category |calories|fat_g|carbs_g|fiber_g|protein_g|sodium_mg|sugar_g|brand     |kind|
+-------------------------------------------------------------+---------+--------+-----+-------+-------+---------+---------+-------+----------+----+
|Big Breakfast (Regular Biscuit)                              |Breakfast|740.0   |48.0 |51.0   |3.0    |28.0     |1560.0   |3.0    |McDonald's|food|
|Big Breakfast with Hotcakes (Regular Biscuit)                |Breakfast|1090.0  |56.0 |111.0  |6.0    |36.0     |2150.0   |17.0   |McDonald's|food|
|Steak & Egg McMuffin                                         |Breakfast|430.0   |23.0 |31.0   |4.0    |26.0     |960.0    |3.0    |McDonald's|food|
|Sausage, Egg & Cheese McGriddles with Egg Whites             |Breakfast|500.0   |26.0 |46

In [ ]:
# Count rows by brand and kind
all_nutrition_df.groupBy("brand", "kind").count().show()

# Count null values in each column
count_nulls(all_nutrition_df)

# Check exact duplicate rows
print("Total rows:", all_nutrition_df.count())
print("Distinct rows:", all_nutrition_df.distinct().count())

In [ ]:
# checking if there are any duplicates in the combined dataframe
all_nutrition_df.groupBy("brand", "item").count() \
    .filter(f.col("count") > 1) \
    .orderBy(f.col("count").desc()) \
    .show(10, truncate=False)

In [49]:
def check_avg(df):
    summary_df = (
        df
        .groupBy("kind", "brand")
        .agg(
            f.count("*").alias("num_items"),
            f.round(f.avg("calories"), 2).alias("avg_calories"),
            f.round(f.avg("fat_g"), 2).alias("avg_fat_g"),
            f.round(f.avg("carbs_g"), 2).alias("avg_carbs_g"),
            f.round(f.avg("fiber_g"), 2).alias("avg_fiber_g"),
            f.round(f.avg("protein_g"), 2).alias("avg_protein_g")
        )
        .orderBy("kind", "brand")
    )
    return summary_df

summary_df = check_avg(all_nutrition_df)
summary_df.show(truncate=False)

+-----+----------+---------+------------+---------+-----------+-----------+-------------+
|kind |brand     |num_items|avg_calories|avg_fat_g|avg_carbs_g|avg_fiber_g|avg_protein_g|
+-----+----------+---------+------------+---------+-----------+-----------+-------------+
|drink|McDonald's|150      |299.47      |7.73     |50.27      |0.77       |7.88         |
|drink|Starbucks |75       |136.87      |2.67     |23.93      |0.55       |4.73         |
|food |McDonald's|110      |462.09      |22.94    |43.36      |2.8        |20.78        |
|food |Starbucks |113      |356.64      |16.35    |41.49      |2.85       |11.47        |
+-----+----------+---------+------------+---------+-----------+-----------+-------------+



In [51]:
def check_median(df):
    summary_df = (
        df
        .groupBy("kind", "brand")
        .agg(
            f.count("*").alias("num_items"),
            f.round(f.median("calories"), 2).alias("median_calories"),
            f.round(f.median("fat_g"), 2).alias("median_fat_g"),
            f.round(f.median("carbs_g"), 2).alias("median_carbs_g"),
            f.round(f.median("fiber_g"), 2).alias("median_fiber_g"),
            f.round(f.median("protein_g"), 2).alias("median_protein_g")
        )
        .orderBy("kind", "brand")
    )
    return summary_df

summary_df = check_median(all_nutrition_df)
summary_df.show(truncate=False)

+-----+----------+---------+---------------+------------+--------------+--------------+----------------+
|kind |brand     |num_items|median_calories|median_fat_g|median_carbs_g|median_fiber_g|median_protein_g|
+-----+----------+---------+---------------+------------+--------------+--------------+----------------+
|drink|McDonald's|150      |270.0          |4.5         |49.0          |1.0           |9.0             |
|drink|Starbucks |75       |130.0          |0.0         |24.0          |0.0           |1.0             |
|food |McDonald's|110      |445.0          |22.0        |42.0          |3.0           |20.0            |
|food |Starbucks |113      |360.0          |17.0        |42.0          |2.0           |8.0             |
+-----+----------+---------+---------------+------------+--------------+--------------+----------------+



In [39]:
all_nutrition_df = all_nutrition_df.select(
    "*",
    f.when(
        f.col("calories") > 0,
        (f.col("protein_g") / f.col("calories") * 100)
    ).otherwise(None).alias("protein_per_100kcal"),

    f.when(
        f.col("calories") > 0,
        (f.col("fiber_g") / f.col("calories") * 100)
    ).otherwise(None).alias("fiber_per_100kcal"),

    f.when(
        f.col("calories") > 0,
        ((f.col("fat_g") * 9) / f.col("calories") * 100)
    ).otherwise(None).alias("fat_pct_cal")
)

all_nutrition_df.show()

+--------------------+---------+--------+-----+-------+-------+---------+---------+-------+----------+----+-------------------+-------------------+------------------+
|                item| category|calories|fat_g|carbs_g|fiber_g|protein_g|sodium_mg|sugar_g|     brand|kind|protein_per_100kcal|  fiber_per_100kcal|       fat_pct_cal|
+--------------------+---------+--------+-----+-------+-------+---------+---------+-------+----------+----+-------------------+-------------------+------------------+
|Big Breakfast (Re...|Breakfast|   740.0| 48.0|   51.0|    3.0|     28.0|   1560.0|    3.0|McDonald's|food|  3.783783783783784|0.40540540540540543| 58.37837837837838|
|Big Breakfast wit...|Breakfast|  1090.0| 56.0|  111.0|    6.0|     36.0|   2150.0|   17.0|McDonald's|food|  3.302752293577982| 0.5504587155963303| 46.23853211009175|
|Steak & Egg McMuffin|Breakfast|   430.0| 23.0|   31.0|    4.0|     26.0|    960.0|    3.0|McDonald's|food|  6.046511627906977| 0.9302325581395349| 48.13953488372093

In [ ]:
# checking protien, fiber per 100 kcal for fair comparison
summary_df = (
        all_nutrition_df
        .groupBy("kind", "brand")
        .agg(
            f.count("*").alias("num_items"),
            f.round(f.avg("protein_per_100kcal"), 2).alias("avg_protein_per_100kcal"),
            f.round(f.avg("fiber_per_100kcal"), 2).alias("avg_fiber_per_100kcal"),
            f.round(f.avg("fat_pct_cal"), 2).alias("avg_fat_pct_cal"),
        )
        .orderBy("kind", "brand")
    )

summary_df.show(truncate=False)

+-----+----------+---------+-----------------------+---------------------+---------------+
|kind |brand     |num_items|avg_protein_per_100kcal|avg_fiber_per_100kcal|avg_fat_pct_cal|
+-----+----------+---------+-----------------------+---------------------+---------------+
|drink|McDonald's|150      |2.69                   |0.28                 |19.78          |
|drink|Starbucks |75       |3.63                   |0.23                 |11.48          |
|food |McDonald's|110      |4.41                   |0.71                 |41.77          |
|food |Starbucks |113      |3.08                   |0.82                 |40.5           |
+-----+----------+---------+-----------------------+---------------------+---------------+



In [ ]:
# comparing sodium content of drinks as no sodium data is available for starbuck's food
all_nutrition_df.groupBy("brand", "kind").agg(
    f.round(f.avg(f.col("sodium_mg")), 2).alias("avg_sodium_mg")
).show()

+----------+-----+-------------+
|     brand| kind|avg_sodium_mg|
+----------+-----+-------------+
|McDonald's| food|       996.64|
|McDonald's|drink|       128.43|
| Starbucks|drink|         64.0|
| Starbucks| food|         NULL|
+----------+-----+-------------+

